# Linear Regression From Scratch

This notebook builds linear regression from first principles. The goal is not just to fit a line, but to understand how the model, loss function, gradients, and optimization process work together.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Generate synthetic data

We start from a simple relationship:

$$y = 3x + 5 + \epsilon$$

where $\epsilon$ is random noise. The model will not know the true values 3 and 5; it has to learn them.

In [ ]:
np.random.seed(42)
x = np.linspace(0, 10, 80)
noise = np.random.normal(0, 2.0, size=len(x))
y = 3 * x + 5 + noise

plt.figure(figsize=(8, 5))
plt.scatter(x, y)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Regression Data')
plt.show()

## 2. Define the model and loss

Our prediction is

$$\hat{y} = wx + b$$

and we measure error using mean squared error:

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

In [ ]:
def predict(x, w, b):
    return w * x + b

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

## 3. Derive the gradients

For the model $\hat{y}=wx+b$, the gradients of MSE are:

$$\frac{\partial MSE}{\partial w} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)x_i$$

$$\frac{\partial MSE}{\partial b} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)$$

These gradients tell us how changing $w$ and $b$ will change the loss.

In [ ]:
def gradients(x, y, w, b):
    y_pred = predict(x, w, b)
    error = y_pred - y
    dw = 2 * np.mean(error * x)
    db = 2 * np.mean(error)
    return dw, db

## 4. Train with gradient descent

We update the parameters using:

$$w \leftarrow w - \eta \frac{\partial MSE}{\partial w}$$
$$b \leftarrow b - \eta \frac{\partial MSE}{\partial b}$$

where $\eta$ is the learning rate.

In [ ]:
w = 0.0
b = 0.0
learning_rate = 0.01
epochs = 2000

loss_history = []

for epoch in range(epochs):
    y_pred = predict(x, w, b)
    loss = mse(y, y_pred)
    loss_history.append(loss)

    dw, db = gradients(x, y, w, b)
    w -= learning_rate * dw
    b -= learning_rate * db

print(f'Learned slope (w): {w:.3f}')
print(f'Learned intercept (b): {b:.3f}')
print(f'Final MSE: {loss_history[-1]:.3f}')

## 5. Visualize the fitted line

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(x, y, label='Data')
plt.plot(x, predict(x, w, b), label='Learned regression line')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Linear Regression Fit')
plt.legend()
plt.show()

## 6. Visualize the loss curve

A decreasing loss indicates that gradient descent is moving the parameters toward a better fit.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training Loss Over Time')
plt.show()

## 7. Compare with scikit-learn

The model above was trained manually. We can compare its result with scikit-learn's implementation.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(x.reshape(-1, 1), y)

print(f'sklearn slope: {model.coef_[0]:.3f}')
print(f'sklearn intercept: {model.intercept_:.3f}')

## 8. Optional experiment: learning rate

Try changing the learning rate. A very small learning rate converges slowly, while a rate that is too large can make training unstable.

In [ ]:
def train_with_lr(lr, epochs=300):
    w_temp, b_temp = 0.0, 0.0
    losses = []
    for _ in range(epochs):
        y_pred = predict(x, w_temp, b_temp)
        losses.append(mse(y, y_pred))
        dw, db = gradients(x, y, w_temp, b_temp)
        w_temp -= lr * dw
        b_temp -= lr * db
    return losses

for lr in [0.001, 0.01, 0.02]:
    losses = train_with_lr(lr)
    plt.plot(losses, label=f'lr={lr}')

plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Effect of Learning Rate')
plt.legend()
plt.show()

## Takeaway

Linear regression can be viewed as an optimization problem. The model has parameters, the loss measures error, and gradient descent updates the parameters in the direction that reduces that error. This same overall framework appears again in more complex machine learning models, including neural networks.